In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/pre_cell_6.pickle

In [4]:
%%RecordEvent
%%time
### cell 6 ###
# Compute correlation on GPU by normalization and dot product
def cell_6():
    # select only numeric columns (returns a cudf DataFrame)
    num_df = df.select_dtypes(include=['number'])
    # compute per‐column mean and unbiased std on GPU
    mean = num_df.mean()
    std = num_df.std(ddof=1)
    # z-score normalization using GPU-native sub/div to avoid Python-operator fallback
    normed = num_df.sub(mean, axis=1).div(std, axis=1)
    # number of rows
    n = len(num_df)
    # correlation matrix via dot product, scaled by 1/(n-1); all ops stay on GPU
    return normed.T.dot(normed).mul(1.0 / (n - 1))

_ = cell_6()

CPU times: user 2min 24s, sys: 3.33 s, total: 2min 28s
Wall time: 2min 27s


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10_small/checkpoints/post_cell_6_try_7.pickle